In [18]:
%%writefile bench_serving.py
"""Streaming latency/throughput benchmark for vLLM and the HealthIQ API.
Targets
  llm  POST {url}/chat/completions (stream=True) — any OpenAI-compatible
       server, e.g. `vllm serve`. Measures model serving.
  app  POST {url}/query/stream (HealthIQ SSE) — measures end to end:
       embedding + FAISS retrieval + prompt build + LLM stream + SSE.

Metrics per concurrency level
  TTFT   time to first content chunk
  ITL    inter-chunk latency (vLLM streams ~1 token per chunk)
  TPOT   (e2e - ttft) / (output_tokens - 1), per request
  E2E    request start to final byte
  throughput: requests/s and output tokens/s over the level's wall time
  errors: count and rate (HTTP errors, timeouts, broken streams, app `error` events)

Every result file records hardware, model, workload hash, and settings so a
run can be reproduced and compared. Example:

  python -m benchmarks.bench_serving --target llm --url http://localhost:8001/v1 \\
      --concurrency 1,4,16 --num-requests 64 --max-tokens 128 --label vllm-l4
"""
from __future__ import annotations
import argparse
import asyncio
import hashlib
import json
import os
import platform
import statistics
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import httpx

HERE = Path(__file__).parent
DEFAULT_WORKLOAD = HERE / "workloads" / "healthcare_queries.jsonl"
SYSTEM_PROMPT = ("You are a healthcare infrastructure analyst. Answer concisely "
                 "using general knowledge; do not invent specific statistics.")


@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None
    e2e_s: Optional[float] = None
    chunk_gaps_s: List[float] = field(default_factory=list)
    output_tokens: int = 0
    token_count_source: str = "chunks"
    error: Optional[str] = None
    server_retrieval_ms: Optional[float] = None


def load_workload(path: Path) -> List[str]:
    return [json.loads(line)["query"] for line in path.read_text().splitlines() if line.strip()]


async def _iter_sse(resp: httpx.Response):
    """Yields (event, data_str) from an SSE response."""
    event, data = "message", []
    async for line in resp.aiter_lines():
        if line == "":
            if data:
                yield event, "\n".join(data)
            event, data = "message", []
        elif line.startswith("event:"):
            event = line[6:].strip()
        elif line.startswith("data:"):
            data.append(line[5:].strip())
    if data:
        yield event, "\n".join(data)


async def one_llm_request(client: httpx.AsyncClient, url: str, model: str, prompt: str,
                          max_tokens: int) -> RequestResult:
    body = {"model": model, "stream": True, "max_tokens": max_tokens, "temperature": 0.0,
            "stream_options": {"include_usage": True},
            "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": prompt}]}
    r = RequestResult(ok=False)
    t0 = time.perf_counter()
    last = None
    try:
        async with client.stream("POST", f"{url}/chat/completions", json=body) as resp:
            if resp.status_code != 200:
                r.error = f"HTTP {resp.status_code}"
                return r
            done = False
            async for _, data in _iter_sse(resp):
                if data == "[DONE]":
                    done = True
                    break
                obj = json.loads(data)
                if obj.get("usage") and obj["usage"].get("completion_tokens") is not None:
                    r.output_tokens = obj["usage"]["completion_tokens"]
                    r.token_count_source = "usage"
                choices = obj.get("choices") or []
                if choices and choices[0].get("delta", {}).get("content"):
                    now = time.perf_counter()
                    if r.ttft_s is None:
                        r.ttft_s = now - t0
                    else:
                        r.chunk_gaps_s.append(now - last)
                    last = now
                    if r.token_count_source == "chunks":
                        r.output_tokens += 1
            if not done:
                r.error = "stream ended without [DONE]"
                return r
    except (httpx.HTTPError, json.JSONDecodeError) as e:
        r.error = type(e).__name__
        return r
    r.e2e_s = time.perf_counter() - t0
    r.ok = r.ttft_s is not None
    if not r.ok:
        r.error = "no content"
    return r


async def one_app_request(client: httpx.AsyncClient, url: str, query: str,
                          max_results: int) -> RequestResult:
    r = RequestResult(ok=False)
    t0 = time.perf_counter()
    last = None
    try:
        async with client.stream("POST", f"{url}/query/stream",
                                 json={"query": query, "max_results": max_results}) as resp:
            if resp.status_code != 200:
                r.error = f"HTTP {resp.status_code}"
                return r
            got_done = False
            async for event, data in _iter_sse(resp):
                obj = json.loads(data)
                if event == "token":
                    now = time.perf_counter()
                    if r.ttft_s is None:
                        r.ttft_s = now - t0
                    else:
                        r.chunk_gaps_s.append(now - last)
                    last = now
                    r.output_tokens += 1
                elif event == "error":
                    r.error = "app error event: " + str(obj.get("message"))[:80]
                elif event == "done":
                    got_done = True
                    r.server_retrieval_ms = obj.get("retrieval_ms")
                    if obj.get("fallback") and not r.error:
                        r.error = "llm fallback"
            if not got_done:
                r.error = r.error or "stream ended without done"
                return r
    except (httpx.HTTPError, json.JSONDecodeError) as e:
        r.error = type(e).__name__
        return r
    r.e2e_s = time.perf_counter() - t0
    r.ok = r.error is None and r.ttft_s is not None
    return r


def _pct(xs: List[float], p: float) -> Optional[float]:
    if not xs:
        return None
    xs = sorted(xs)
    k = (len(xs) - 1) * p / 100
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


def _dist_ms(xs: List[float]) -> Dict[str, Optional[float]]:
    if not xs:
        return {"n": 0}
    ms = [x * 1000 for x in xs]
    return {"n": len(ms), "mean": round(statistics.fmean(ms), 2),
            **{f"p{p}": round(_pct(ms, p), 2) for p in (50, 90, 99)}}


async def run_level(args, prompts: List[str], concurrency: int, model: str) -> Dict[str, Any]:
    sem = asyncio.Semaphore(concurrency)
    timeout = httpx.Timeout(args.timeout_s, connect=10.0)
    limits = httpx.Limits(max_connections=concurrency, max_keepalive_connections=concurrency)
    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        async def task(i: int) -> RequestResult:
            p = prompts[i % len(prompts)]
            async with sem:
                if args.target == "llm":
                    return await one_llm_request(client, args.url, model, p, args.max_tokens)
                return await one_app_request(client, args.url, p, args.max_results)

        for i in range(args.warmup):  # warmup: sequential, not recorded
            await task(i)
        t0 = time.perf_counter()
        results = await asyncio.gather(*(task(i) for i in range(args.num_requests)))
        wall = time.perf_counter() - t0

    ok = [r for r in results if r.ok]
    tpot = [(r.e2e_s - r.ttft_s) / (r.output_tokens - 1) for r in ok if r.output_tokens > 1]
    errors: Dict[str, int] = {}
    for r in results:
        if not r.ok:
            errors[r.error or "unknown"] = errors.get(r.error or "unknown", 0) + 1
    out_tokens = sum(r.output_tokens for r in ok)
    retrieval = [r.server_retrieval_ms / 1000 for r in ok if r.server_retrieval_ms is not None]
    return {
        "concurrency": concurrency,
        "num_requests": len(results),
        "succeeded": len(ok),
        "error_rate": round(1 - len(ok) / len(results), 4) if results else None,
        "errors": errors,
        "wall_time_s": round(wall, 3),
        "request_throughput_rps": round(len(ok) / wall, 3) if wall else None,
        "output_token_throughput_tps": round(out_tokens / wall, 2) if wall else None,
        "output_tokens_total": out_tokens,
        "token_count_source": sorted({r.token_count_source for r in ok}),
        "ttft_ms": _dist_ms([r.ttft_s for r in ok]),
        "itl_ms": _dist_ms([g for r in ok for g in r.chunk_gaps_s]),
        "tpot_ms": _dist_ms(tpot),
        "e2e_ms": _dist_ms([r.e2e_s for r in ok]),
        **({"server_retrieval_ms": _dist_ms(retrieval)} if retrieval else {}),
    }


def _cmd(cmd: List[str]) -> Optional[str]:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=10).stdout.strip() or None
    except (OSError, subprocess.SubprocessError):
        return None


def hardware_info(gpu_note: Optional[str]) -> Dict[str, Any]:
    info = {"client_platform": platform.platform(), "client_machine": platform.machine(),
            "client_cpu_count": os.cpu_count(), "python": sys.version.split()[0]}
    if sys.platform == "darwin":
        info["client_cpu"] = _cmd(["sysctl", "-n", "machdep.cpu.brand_string"])
        mem = _cmd(["sysctl", "-n", "hw.memsize"])
        info["client_mem_gb"] = round(int(mem) / 2**30, 1) if mem else None
    info["local_nvidia_smi"] = _cmd(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                                     "--format=csv,noheader"])
    # The server may be remote: say what it runs on, since the client can't see it.
    info["server_hardware_note"] = gpu_note
    return info


def discover_model(args) -> Dict[str, Any]:
    try:
        if args.target == "llm":
            data = httpx.get(f"{args.url}/models", timeout=10).json()
            return {"model": args.model or data["data"][0]["id"],
                    "served_models": [m["id"] for m in data["data"]]}
        ready = httpx.get(f"{args.url}/ready", timeout=10).json()
        return {"model": ready.get("llm", {}).get("model"), "app_llm": ready.get("llm"),
                "hospitals_loaded": ready.get("hospitals_loaded")}
    except Exception as e:  # metadata only; the benchmark itself will surface real failures
        return {"model": args.model, "discovery_error": str(e)}


def _git_commit() -> Optional[str]:
    return _cmd(["git", "-C", str(HERE.parent), "rev-parse", "--short", "HEAD"])


def print_table(levels: List[Dict[str, Any]]) -> None:
    hdr = f"{'conc':>4} {'ok/n':>7} {'err%':>5} {'req/s':>7} {'tok/s':>8} " \
          f"{'TTFT p50':>9} {'TTFT p99':>9} {'ITL p50':>8} {'TPOT p50':>9} {'E2E p50':>8} {'E2E p99':>8}"
    print(hdr)
    for lv in levels:
        g = lambda k, p: lv[k].get(p) if lv[k].get("n") else None  # noqa: E731
        f = lambda v: f"{v:.1f}" if v is not None else "-"  # noqa: E731
        print(f"{lv['concurrency']:>4} {lv['succeeded']:>3}/{lv['num_requests']:<3} "
              f"{lv['error_rate'] * 100:>5.1f} {lv['request_throughput_rps']:>7.2f} "
              f"{lv['output_token_throughput_tps']:>8.1f} {f(g('ttft_ms', 'p50')):>9} "
              f"{f(g('ttft_ms', 'p99')):>9} {f(g('itl_ms', 'p50')):>8} {f(g('tpot_ms', 'p50')):>9} "
              f"{f(g('e2e_ms', 'p50')):>8} {f(g('e2e_ms', 'p99')):>8}")


def main(argv: Optional[List[str]] = None) -> Dict[str, Any]:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--target", choices=["llm", "app"], required=True)
    ap.add_argument("--url", required=True, help="llm: base incl. /v1; app: API root")
    ap.add_argument("--model", help="llm target: model name (default: first from /v1/models)")
    ap.add_argument("--workload", type=Path, default=DEFAULT_WORKLOAD)
    ap.add_argument("--concurrency", default="1,4,16")
    ap.add_argument("--num-requests", type=int, default=32)
    ap.add_argument("--warmup", type=int, default=2)
    ap.add_argument("--max-tokens", type=int, default=128)
    ap.add_argument("--max-results", type=int, default=10, help="app target: retrieval top_k")
    ap.add_argument("--timeout-s", type=float, default=120.0)
    ap.add_argument("--label", default="run")
    ap.add_argument("--gpu-note", help="server hardware, e.g. '1x NVIDIA L4 24GB, driver 550'")
    ap.add_argument("--out-dir", type=Path, default=HERE / "results")
    args = ap.parse_args(argv)
    args.url = args.url.rstrip("/")

    prompts = load_workload(args.workload)
    model_meta = discover_model(args)
    model = model_meta.get("model") or args.model or "unknown"
    levels = []
    for c in (int(x) for x in args.concurrency.split(",")):
        lv = asyncio.run(run_level(args, prompts, c, model))
        levels.append(lv)
    report = {
        "label": args.label,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "git_commit": _git_commit(),
        "target": args.target,
        "url": args.url,
        "model": model_meta,
        "hardware": hardware_info(args.gpu_note),
        "workload": {"file": str(args.workload.relative_to(HERE.parent)) if args.workload.is_relative_to(HERE.parent) else str(args.workload),
                     "sha256": hashlib.sha256(args.workload.read_bytes()).hexdigest()[:16],
                     "num_prompts": len(prompts), "max_tokens": args.max_tokens,
                     "max_results": args.max_results, "temperature": 0.0,
                     "warmup": args.warmup, "num_requests_per_level": args.num_requests},
        "levels": levels,
    }
    args.out_dir.mkdir(parents=True, exist_ok=True)
    out = args.out_dir / f"{args.label}.json"
    out.write_text(json.dumps(report, indent=2))
    print(f"[{args.label}] target={args.target} model={model}")
    print_table(levels)
    print(f"wrote {out}")
    return report


if __name__ == "__main__":
    main()

Overwriting bench_serving.py


In [19]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!find /kaggle/input -name "healthcare_queries.txt"
!ls -la bench_serving.py

Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
/kaggle/input/datasets/dhyanagni/healthiq-kaggle/healthcare_queries.txt
-rw-r--r-- 1 root root 14193 Oct  8 03:29 bench_serving.py


In [5]:
!pip install -q vllm httpx
!CUDA_VISIBLE_DEVICES=0 nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8001 --dtype half \
    --max-model-len 4096 --gpu-memory-utilization 0.90 > vllm.log 2>&1 &
!for i in $(seq 1 120); do curl -sf localhost:8001/health && echo READY && break; sleep 5; done; tail -3 vllm.log

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 5.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 66.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 90.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 46.3 MB/s eta 0:00:00:00:0100:01


OSError: Background processes not supported.

In [20]:
import subprocess, os, time, urllib.request

log = open("vllm.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct", "--port", "8001", "--dtype", "half",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
print("started, pid", proc.pid)

for i in range(120):
    if proc.poll() is not None:
        print("vLLM EXITED with code", proc.returncode); break
    try:
        urllib.request.urlopen("http://localhost:8001/health", timeout=2); print("READY"); break
    except Exception:
        time.sleep(5)

print(open("vllm.log").read()[-2500:])

started, pid 673
READY
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [21]:
!pip uninstall -y -q torchaudio
!nvidia-smi --query-gpu=driver_version --format=csv,noheader
import subprocess
print(subprocess.run(["python", "-c",
    "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, "
    "'available', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    capture_output=True, text=True).stdout)

580.178.04
580.178.04
torch 2.13.0+cu130 cuda 13.0 available True Tesla T4



In [22]:
Q = "/kaggle/input/datasets/dhyanagni/healthiq-kaggle/healthcare_queries.txt"
!python bench_serving.py --target llm --url http://localhost:8001/v1 \
    --workload {Q} --out-dir /kaggle/working \
    --concurrency 1,4,8,16,32 --num-requests 64 --max-tokens 128 --label vllm-t4-qwen1.5b \
    --gpu-note "$(nvidia-smi -i 0 --query-gpu=name,memory.total,driver_version --format=csv,noheader) (1 of 2 T4s, CUDA_VISIBLE_DEVICES=0); vllm $(pip show vllm | grep Version); torch 2.13.0+cu130"

[vllm-t4-qwen1.5b] target=llm model=Qwen/Qwen2.5-1.5B-Instruct
conc    ok/n  err%   req/s    tok/s  TTFT p50  TTFT p99  ITL p50  TPOT p50  E2E p50  E2E p99
   1  64/64    0.0    0.59     53.3      40.1      47.4     18.4      18.3   1640.3   2620.8
   4  64/64    0.0    2.59    233.7      47.4      55.8     15.7      15.8   1364.7   2089.3
   8  64/64    0.0    4.84    433.0      51.9      72.9     16.3      16.5   1393.3   2163.7
  16  64/64    0.0    8.10    726.0      57.0      94.8     17.9      18.4   1550.1   2404.3
  32  64/64    0.0   11.76   1058.5      95.0     162.2     23.2      23.4   2045.4   3142.0
wrote /kaggle/working/vllm-t4-qwen1.5b.json


In [23]:
!pip show vllm | grep Version
!grep -i -E "attention backend|Using .*backend|KV cache|GPU KV cache size|Maximum concurrency|cuda graph|max_num_seqs|dtype" vllm.log | head -15

Version: 0.31.0
grep: vllm.log: binary file matches


In [25]:
# RAG-sized vLLM sweep for a Kaggle/Colab GPU notebook — paste as ONE cell.
#
# Prereqs in the notebook: vLLM installed (torchaudio removed), bench_serving.py
# written to /kaggle/working (via %%writefile), and rag_prompts.txt attached as
# a dataset (generated from benchmarks/workloads/healthcare_rag_prompts.jsonl:
# real HealthIQ RAG prompts for the 24 benchmark queries).
#
# What it does:
#   1. restarts vLLM with prefix caching OFF (repeated long prompts would
#      otherwise hide prefill cost),
#   2. measures real prompt lengths with vLLM's own tokenizer (/tokenize),
#   3. sweeps concurrency 1..128, 3 repeats, >= 2x concurrency requests per level,
#   4. prints mean ± std across repeats and zips everything for download.
import glob, json, os, shutil, statistics, subprocess, sys, time, urllib.request

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
URL = "http://localhost:8001"
LEVELS = [1, 4, 8, 16, 32, 64, 128]
REPS = 3
OUT = "/kaggle/working/rag_sweep"
os.makedirs(OUT, exist_ok=True)

RAG = glob.glob("/kaggle/input/**/rag_prompts.txt", recursive=True)
assert RAG, "rag_prompts.txt not found under /kaggle/input — attach the dataset first"
RAG = RAG[0]
assert os.path.exists("bench_serving.py"), "bench_serving.py missing — rerun the %%writefile cell"
print("workload:", RAG)


def get(path, data=None, timeout=5):
    req = urllib.request.Request(URL + path, data=json.dumps(data).encode() if data else None,
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())


# 1. restart vLLM without prefix caching
try:
    proc.terminate(); proc.wait(timeout=60)  # noqa: F821 — from the earlier start cell, if present
except Exception:
    pass
subprocess.run(["pkill", "-f", "vllm"])  # also stops a lingering EngineCore child
for _ in range(60):  # wait until GPU 0 memory is actually released
    used = subprocess.run(["nvidia-smi", "-i", "0", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip()
    if used.isdigit() and int(used) < 1000:
        break
    time.sleep(2)
print("GPU 0 memory used before restart (MiB):", used)
log = open(f"{OUT}/vllm_nocache.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8001", "--dtype", "half", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--no-enable-prefix-caching"],
    stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
t0 = time.time()
for _ in range(120):
    if proc.poll() is not None:
        print(open(f"{OUT}/vllm_nocache.log").read()[-3000:])
        raise SystemExit(f"vLLM exited with code {proc.returncode}")
    try:
        get("/health", timeout=2); break
    except Exception:
        time.sleep(5)
print(f"vLLM ready in {time.time() - t0:.0f}s")
cfg = open(f"{OUT}/vllm_nocache.log").read()
print("prefix caching disabled:", "enable_prefix_caching=False" in cfg)

# 2. real prompt lengths, counted by vLLM's tokenizer (system prompt of the harness not included)
prompts = [json.loads(l)["query"] for l in open(RAG) if l.strip()]
try:
    counts = sorted(get("/tokenize", {"model": MODEL, "prompt": p})["count"] for p in prompts)
    prompt_stats = {"n": len(counts), "min": counts[0], "median": counts[len(counts) // 2], "max": counts[-1],
                    "source": "vllm /tokenize"}
except Exception as e:  # non-fatal: fall back to a rough chars/4 estimate
    counts = sorted(len(p) // 4 for p in prompts)
    prompt_stats = {"n": len(counts), "min": counts[0], "median": counts[len(counts) // 2], "max": counts[-1],
                    "source": f"estimate chars/4 (/tokenize failed: {e})"}
print("RAG prompt tokens:", prompt_stats)

def _out(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return ""


VER = next((l.split()[-1] for l in _out([sys.executable, "-m", "pip", "show", "vllm"]).splitlines()
            if l.startswith("Version")), "?")
GPU = _out(["nvidia-smi", "-i", "0", "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader"]) or "unknown GPU"
NOTE = (f"{GPU} (1 of 2, CUDA_VISIBLE_DEVICES=0); vllm {VER}; fp16; prefix caching OFF; "
        f"RAG prompts median {prompt_stats['median']} tokens")

# 3. sweep
results = {c: [] for c in LEVELS}
start = time.time()
for rep in range(1, REPS + 1):
    for c in LEVELS:
        n = max(32, 2 * c)
        label = f"vllm-t4-rag-c{c}-r{rep}"
        r = subprocess.run([sys.executable, "bench_serving.py", "--target", "llm", "--url", URL + "/v1",
                            "--workload", RAG, "--concurrency", str(c), "--num-requests", str(n),
                            "--warmup", "1", "--max-tokens", "128", "--timeout-s", "300",
                            "--label", label, "--out-dir", OUT, "--gpu-note", NOTE],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-1500:], r.stderr[-1500:]); raise SystemExit(f"benchmark failed at c={c}")
        lv = json.load(open(f"{OUT}/{label}.json"))["levels"][0]
        results[c].append(lv)
        print(f"rep {rep} c={c:>3}  ok {lv['succeeded']:>3}/{lv['num_requests']:<3} "
              f"tok/s {lv['output_token_throughput_tps']:>7.1f}  TTFT p50 {lv['ttft_ms'].get('p50', float('nan')):>7.1f} ms  "
              f"ITL p50 {lv['itl_ms'].get('p50', float('nan')):>5.1f} ms   [{(time.time() - start) / 60:.1f} min]", flush=True)


# 4. mean ± std across repeats
def ms(c, key, stat="p50"):
    xs = [lv[key].get(stat) for lv in results[c] if lv[key].get(stat) is not None]
    return (statistics.fmean(xs), statistics.stdev(xs) if len(xs) > 1 else 0.0) if xs else (float("nan"), 0.0)


summary = {"note": NOTE, "model": MODEL, "vllm": VER, "prompt_tokens": prompt_stats,
           "workload": os.path.basename(RAG), "max_tokens": 128, "reps": REPS, "levels": {}}
print(f"\nmean ± std over {REPS} repeats — RAG prompts (median {prompt_stats['median']} tokens), prefix caching OFF")
print(f"{'conc':>4} {'n/rep':>5} {'err':>4} {'tok/s':>15} {'req/s':>12} {'TTFT p50 ms':>15} {'TTFT p99 ms':>15} {'ITL p50 ms':>12} {'E2E p50 ms':>15}")
for c in LEVELS:
    tps = [lv["output_token_throughput_tps"] for lv in results[c]]
    rps = [lv["request_throughput_rps"] for lv in results[c]]
    errs = sum(lv["num_requests"] - lv["succeeded"] for lv in results[c])
    row = {"tok_s": (statistics.fmean(tps), statistics.stdev(tps)), "req_s": (statistics.fmean(rps), statistics.stdev(rps)),
           "ttft_p50": ms(c, "ttft_ms"), "ttft_p99": ms(c, "ttft_ms", "p99"),
           "itl_p50": ms(c, "itl_ms"), "e2e_p50": ms(c, "e2e_ms"), "errors": errs,
           "requests_per_rep": results[c][0]["num_requests"]}
    summary["levels"][c] = row
    f = lambda t: f"{t[0]:.1f} ± {t[1]:.1f}"  # noqa: E731
    print(f"{c:>4} {row['requests_per_rep']:>5} {errs:>4} {f(row['tok_s']):>15} {f(row['req_s']):>12} "
          f"{f(row['ttft_p50']):>15} {f(row['ttft_p99']):>15} {f(row['itl_p50']):>12} {f(row['e2e_p50']):>15}")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2)
shutil.make_archive("/kaggle/working/vllm-t4-rag-sweep", "zip", OUT)
print(f"\ndone in {(time.time() - start) / 60:.1f} min -> download /kaggle/working/vllm-t4-rag-sweep.zip")


workload: /kaggle/input/datasets/dhyanagni/rag-prompt/rag_prompts.txt
GPU 0 memory used before restart (MiB): 0
vLLM ready in 600s
prefix caching disabled: True
RAG prompt tokens: {'n': 24, 'min': 449, 'median': 536, 'max': 772, 'source': 'vllm /tokenize'}
rep 1 c=  1  ok  32/32  tok/s    53.6  TTFT p50   154.8 ms  ITL p50  17.1 ms   [1.3 min]
rep 1 c=  4  ok  32/32  tok/s   179.7  TTFT p50   412.4 ms  ITL p50  17.7 ms   [1.8 min]
rep 1 c=  8  ok  32/32  tok/s   271.9  TTFT p50   619.3 ms  ITL p50  20.4 ms   [2.0 min]
rep 1 c= 16  ok  32/32  tok/s   368.2  TTFT p50  1162.3 ms  ITL p50  25.6 ms   [2.3 min]
rep 1 c= 32  ok  64/64  tok/s   441.5  TTFT p50  1377.9 ms  ITL p50  37.5 ms   [2.6 min]
rep 1 c= 64  ok 128/128 tok/s   514.3  TTFT p50  1920.2 ms  ITL p50  55.6 ms   [3.2 min]
rep 1 c=128  ok 256/256 tok/s   556.1  TTFT p50  2169.3 ms  ITL p50  89.4 ms   [4.2 min]
rep 2 c=  1  ok  32/32  tok/s    55.0  TTFT p50   153.6 ms  ITL p50  17.0 ms   [5.5 min]
rep 2 c=  4  ok  32/32  tok/s  